<a href="https://colab.research.google.com/github/VukasinA/ML_projekti/blob/main/NNDom2VA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Domaci 2 Neuronske mreze

In [ ]:
# Инсталација (ако је потребно)
!pip install torch torchvision
!pip install tensorflow

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

PyTorch:

In [ ]:
# Подешавање трансформација
transform = transforms.Compose([transforms.ToTensor()])

# Учитавање података
dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform)

# Подела на тренинг и валидацију
train_dataset, val_dataset = random_split(dataset, [50000, 10000])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

# CNN у PyTorch-у
class CNN_PyTorch(nn.Module):
    def __init__(self):
        super(CNN_PyTorch, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),  # 1 канал (црно-бело), 32 филтера
            nn.ReLU(),
            nn.MaxPool2d(2),                  # Смањује димензију 2x
            nn.Conv2d(32, 64, kernel_size=3), # 64 филтера
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layer = nn.Sequential(
            nn.Flatten(),           # Претвара у 1D вектор
            nn.Linear(64*5*5, 128), # Fully connected: улаз 64x5x5 = 1600
            nn.ReLU(),
            nn.Linear(128, 10)      # Излаз: 10 класа (ципеле, мајице, сл.)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_torch = CNN_PyTorch().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_torch.parameters(), lr=0.001)

# Тренинг петља
def train_pytorch(model, loader):
    model.train()
    for epoch in range(5):
        total_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()  #nulira gradijente
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()  #azurira tezine
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

train_pytorch(model_torch, train_loader)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 278kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.75MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.5MB/s]


Epoch 1, Loss: 438.4543
Epoch 2, Loss: 281.0399
Epoch 3, Loss: 242.3404
Epoch 4, Loss: 216.5128
Epoch 5, Loss: 195.5944


Tenosr flow:

In [ ]:
# Преузимање FashionMNIST сета
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Скалирање и обликавање
x_train = x_train[..., np.newaxis] / 255.0
x_test = x_test[..., np.newaxis] / 255.0

# Подела на тренинг и валидацију
x_train_tf, x_val_tf = x_train[:50000], x_train[50000:]
y_train_tf, y_val_tf = y_train[:50000], y_train[50000:]

# Модел у TensorFlow-у
model_tf = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)), #32 filtera velicine 3x3
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'), #Dublji sloj sa 64 filtera
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_tf.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Тренинг
model_tf.fit(x_train_tf, y_train_tf, epochs=5, validation_data=(x_val_tf, y_val_tf))

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 51s 31ms/step - accuracy: 0.7665 - loss: 0.6404 - val_accuracy: 0.8708 - val_loss: 0.3555
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 78s 28ms/step - accuracy: 0.8833 - loss: 0.3170 - val_accuracy: 0.8830 - val_loss: 0.3111
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 28ms/step - accuracy: 0.9028 - loss: 0.2652 - val_accuracy: 0.8985 - val_loss: 0.2799
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 29ms/step - accuracy: 0.9140 - loss: 0.2282 - val_accuracy: 0.9076 - val_loss: 0.2504
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - accuracy: 0.9258 - loss: 0.1978 - val_accuracy: 0.9032 - val_loss: 0.2720


**Poredjenje PyTorch vs TensorFlow:**

---
Stil koda kod PyTorcha je IMPERATIVAN dok je kod TensorFlowa DEKLARATIVAN

---
Definicija modela: PyTorch - Nasledjivanjem kod nn.Module TensorFlow - Sekvencijalan

---
Obuka: PyTorch - Rucno kontrolisana (optimizer.step())  TensorFlow - Automatizovana (model.fit())

---
Vidljivost tokom izracunavanja: PyTorch - Da TensorFlow - Ne uvek

---
Koriscenje u istrazivanju: PyTorch - Cesce kod istrazivaca TensorFlow - Popularniji u industriji



